In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent.parent))

import numpy as np
import pandas as pd
import mlflow
from src.common.utils.config import load_config
from src.modeling.feature_selector import get_feature_columns
from src.modeling.splitter         import time_split
from src.modeling.metrics          import mape, wmape
from src.modeling.regressor        import train_regressor
from src.modeling.classifier       import train_classifier
from src.modeling.two_stage        import train_two_stage
from src.modeling.model_io         import save_model

In [ ]:
MODEL_NAME = "model_1_monthly_item_continuous_intermittent"
IS_TWO_STAGE = False
print(f"MODEL_NAME: {MODEL_NAME} | IS_TWO_STAGE: {IS_TWO_STAGE}")

In [ ]:
cfg = load_config()
mod_cfg = cfg["modeling"]
matrix_path = "../../data/intermediate/item_monthly_modeling_matrix.parquet"
df = pd.read_parquet(matrix_path)

# Filter for Continuous and Intermittent
df = df[df["demand_class"].isin(["Continuous", "Intermittent"])].copy()

# Ensure targets
df["target_qty_log1p"] = np.log1p(df["target_qty_raw"])
df = df.dropna(subset=["target_qty_raw"])
df = df.reset_index(drop=True)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

In [ ]:
df_train, df_eval = time_split(df, mod_cfg["train_end"], mod_cfg["eval_start"])
print(f"Train: {len(df_train):,} rows ({mod_cfg['train_start']} – {mod_cfg['train_end']})")
print(f"Eval:  {len(df_eval):,} rows ({mod_cfg['eval_start']} – {mod_cfg['eval_end']})")

In [ ]:
target_col = mod_cfg["target_regressor"]
feature_cols, target_col = get_feature_columns(df_train, target_col, mod_cfg["drop_columns"])
print(f"Feature count: {len(feature_cols)}")

In [ ]:
mlflow.set_tracking_uri(mod_cfg["mlflow_tracking_uri"])
mlflow.set_experiment(mod_cfg["mlflow_experiment"])

reg = train_regressor(
    df_train, df_eval, feature_cols, target_col,
    mod_cfg["xgb_param_grid"], cfg, MODEL_NAME
)

In [ ]:
X_eval = df_eval[feature_cols].values
y_true_raw = df_eval["target_qty_raw"].values

y_pred_raw = reg.predict(X_eval)

mape_val, zero_frac = mape(y_true_raw, y_pred_raw)
wmape_val = wmape(y_true_raw, y_pred_raw)
print(f"MAPE  : {mape_val:.2f}%  (zero actuals excluded: {zero_frac:.1%})")
print(f"WMAPE : {wmape_val:.2f}%")

In [ ]:
output_dir = Path(mod_cfg["model_output_path"])
save_model(reg, output_dir, MODEL_NAME)
print(f"Model saved to {output_dir}")